# 🛒 Lab 13-14: Холбоо Хамаарлын Дүн Шинжилгээ (Association Rule Mining)
## dataset_2.csv ашиглан Apriori Алгоритм

---
### 📌 Энэ дасгалын зорилго юу вэ?

**Холбоо хамаарлын дүн шинжилгээ (Association Rule Mining)** гэдэг нь дэлгүүрийн худалдан авалтын өгөгдлөөс **"хэрэглэгчид ямар бүтээгдэхүүнийг хамт хэрэглэж байна"** гэдгийг олж тогтоох аргачлал юм.

Жишээ нь: _"ТАЛХ авсан хүн СҮҮГ ч авах магадлал өндөр"_ гэсэн хэв маягийг автоматаар олдог.

Энэ нь **Apriori алгоритм** ашиглан хийгддэг бөгөөд дэлгүүрүүд ашигт тохируулга, зохион байгуулалт хийхэд ашигладаг.

---
## 📦 АЛХАМ 1: Шаардлагатай номын санг импортлох

**Тайлбар (15 настанд):** Кодын ажиллахад хэрэгтэй "хэрэгслүүдийг" татаж авах юм шиг ойлгоорой. `pandas` нь хүснэгттэй ажиллах, `numpy` нь тоон тооцоолол хийх, `mlxtend` нь Apriori алгоритмыг ашиглах зориулалттай.

In [ ]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

print("✅ Бүх номын сан амжилттай импортлогдлоо!")

---
## 📂 АЛХАМ 2: Өгөгдлийн багцыг уншиж, танилцах

**Тайлбар:** `dataset_2.csv` файлд 20 ширхэг гүйлгээ (transaction) байна. Гүйлгээ бүрт хэрэглэгчийн авсан бүтээгдэхүүнүүд таслалаар тусгаарлагдсан байна.

Жишээ нь: нэг мөр `MILK,BREAD,BISCUIT` гэж байвал тухайн хэрэглэгч **Сүү, Талх, Бисквит** авсан гэсэн үг.

In [ ]:
# Өгөгдлийн багцыг унших - header байхгүй учир header=None
data = pd.read_csv("dataset_2.csv", header=None)
print(f"📊 Нийт гүйлгээний тоо: {data.shape[0]}")
print()
print("Эхний 5 гүйлгээ:")
data.head()

In [ ]:
# Бүх гүйлгээг харах
print("📋 Бүх 20 гүйлгээ:")
for i, row in data.iterrows():
    print(f"  Гүйлгээ {i+1:>2}: {row[0]}")

---
## 🔧 АЛХАМ 3: Өгөгдлийг боловсруулах (Data Preprocessing)

**Тайлбар:** dataset_2.csv-ийн бүтэц dataset_1.csv-ээс ялгаатай. Энд гүйлгээ бүр нэг нүдэнд таслалаар тусгаарлагдсан байна. Тиймээс эхлээд мөр бүрийг тусдаа **жагсаалт (list)** болгон хувиргах хэрэгтэй.

**Preprocessing** = "Өгөгдлийг цэвэрлэж, алгоритмд тохиромжтой хэлбэрт оруулах" гэсэн утгатай.

In [ ]:
# Гүйлгээ бүрийг жагсаалт болгон хувиргах
# Жишээ нь: "MILK,BREAD,BISCUIT" → ['MILK', 'BREAD', 'BISCUIT']

transactions = []
for i in range(data.shape[0]):
    # .split(',') нь таслалаар тусгаарлаж жагсаалт болгодог
    items = data.values[i, 0].split(',')
    # .strip() нь хоосон зайг арилгадаг
    items = [item.strip() for item in items]
    transactions.append(items)

print(f"✅ Нийт {len(transactions)} гүйлгээ боловсруулагдлаа.")
print()
print("Эхний 5 гүйлгээний жагсаалт хэлбэр:")
for i, t in enumerate(transactions[:5]):
    print(f"  Гүйлгээ {i+1}: {t}")

---
## 📊 АЛХАМ 4: Бараа бүтээгдэхүүний давтамж шинжилгээ

**Тайлбар:** Ямар бараа хамгийн их гүйлгээнд тохиолддогийг олж харна. Энэ нь "хамгийн их зардаг бараа" юу болохыг харуулдаг.

In [ ]:
# Бүх барааг нэг жагсаалтад цуглуулах
all_items = []
for transaction in transactions:
    all_items.extend(transaction)

# DataFrame болгон хадгалах
df = pd.DataFrame(all_items, columns=["items"])
df["incident_count"] = 1  # Тоолохын тулд нэмэлт багана

# Бараа бүрийн давтамжийг бүлэглэж тооцох
df_table = df.groupby("items").sum().sort_values("incident_count", ascending=False).reset_index()

print("📦 Бараа бүтээгдэхүүний давтамж (өндрөөс бага руу):")
print()
df_table

In [ ]:
# Нийт өвөрмөц барааны тоо
print(f"🔢 Нийт өвөрмөц бараа бүтээгдэхүүний тоо: {df_table.shape[0]}")
print(f"🔢 Нийт бараа бүртгэл (давтамжтай): {df.shape[0]}")

---
## 🔄 АЛХАМ 5: TransactionEncoder ашиглан өгөгдлийг хөрвүүлэх

**Тайлбар:** `TransactionEncoder` нь жагсаалт хэлбэртэй гүйлгээнүүдийг **True/False (үнэн/худал)** матриц болгон хувиргадаг.

Яагаад? Учир нь Apriori алгоритм зөвхөн 0 ба 1 утгатай өгөгдөлтэй ажиллах чадвартай.

Жишээ нь:
| Гүйлгээ | BREAD | MILK | TEA | COFFEE |
|---------|-------|------|-----|--------|
| 1       | True  | True | False | False |
| 2       | True  | False| True  | False |

In [ ]:
# TransactionEncoder: Гүйлгээний жагсаалтыг матриц болгон хувиргах
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)

# DataFrame болгох
dataset = pd.DataFrame(te_ary, columns=te.columns_)

print(f"📐 Dataset хэмжээ: {dataset.shape[0]} гүйлгээ × {dataset.shape[1]} бараа")
print()
dataset

---
## 🔢 АЛХАМ 6: True/False-ийг 1/0 болгон кодлох

**Тайлбар:** Apriori алгоритмд **True → 1**, **False → 0** болгон хөрвүүлж өгдөг. Энэ нь компьютерт тооцоолол хийхэд илүү хялбар.

In [ ]:
# True/False → 1/0 хөрвүүлэх функц
def encode_units(x):
    if x == False:
        return 0
    if x == True:
        return 1

# applymap ашиглан DataFrame дотрах бүх утгыг хөрвүүлэх
dataset = dataset.applymap(encode_units)

print("✅ Кодлолт дууслаа. Эхний 10 мөр:")
dataset.head(10)

---
## ⚙️ АЛХАМ 7: Apriori алгоритм ажиллуулах

**Тайлбар:** Apriori алгоритм нь **"хамгийн их тохиолддог зүйлсийн бүлгүүдийг" (Frequent Itemsets)** олдог.

### Гол ойлголтууд:
- **Support (Дэмжлэг)** = Тухайн бараа(нуудын хослол) нийт гүйлгээнд хэдэн хувьтай тохиолдож байна?
  - Жишээ: BREAD 20 гүйлгээнээс 12-т тохиолдвол → Support = 12/20 = **0.60 (60%)**
- **min_support=0.05** = Дор хаяж нийт гүйлгээний 5%-д тохиолдох ёстой гэсэн босго утга

> Dataset жижиг (20 гүйлгээ) учир min_support-ийг 0.05 болгон тохируулсан.

In [ ]:
# Apriori алгоритмаар хамгийн их тохиолддог зүйлсийн багцыг олох
# min_support=0.05 → дор хаяж 5% тохиолдох ёстой
# use_colnames=True → баганын нэрийг ашиглах

frequent_itemsets = apriori(dataset, min_support=0.05, use_colnames=True)

# Урт (хэдэн барааны хослол болохыг) нэмэх
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print(f"✅ Нийт {frequent_itemsets.shape[0]} ширхэг давтамжтай зүйлсийн багц олдлоо.")
print()
frequent_itemsets.sort_values('support', ascending=False)

---
## 🔍 АЛХАМ 8: Нэг бараатай зүйлсийн багцыг харах

**Тайлбар:** `length == 1` гэдэг нь зөвхөн ганц бараа агуулсан зүйлсийн багцуудыг харуулна. Эдгээр нь хамгийн их зардаг барааны жагсаалт юм.

In [ ]:
# Ганц бараатай (length == 1) тохиолдлуудыг шүүх
single_items = frequent_itemsets[frequent_itemsets['length'] == 1].sort_values('support', ascending=False)
print("📦 Хамгийн их дэмжлэгтэй (support) нэг бараанаас бүрдсэн зүйлсийн багц:")
single_items

---
## 🔗 АЛХАМ 9: Хоёр бараатай зүйлсийн багцыг харах

**Тайлбар:** `length == 2` гэдэг нь хоёр барааны хослолыг харуулна. Эдгээр нь хамт зардаг бараануудын хослол юм.

In [ ]:
# Хоёр бараатай (length == 2) зүйлсийн багцыг шүүх
two_items = frequent_itemsets[frequent_itemsets['length'] == 2].sort_values('support', ascending=False)
print("🔗 Хоёр бараатай зүйлсийн багц (support >= 5%):")
two_items

---
## 🔗 АЛХАМ 10: Гурван бараатай зүйлсийн багцыг харах

**Тайлбар:** Гурван барааны хослолыг харна. Жишээ нь `{BREAD, MILK, BISCUIT}` гэдэг гурав нь хамт авагддаг гэсэн үг.

In [ ]:
# Гурван бараатай (length == 3) зүйлсийн багцыг харах
three_items = frequent_itemsets[frequent_itemsets['length'] == 3].sort_values('support', ascending=False)
print("🔗 Гурван бараатай зүйлсийн багц:")
if three_items.empty:
    print("⚠️ Гурван бараатай давтамжтай зүйлсийн багц олдсонгүй.")
    print("   (Ихэвчлэн dataset жижиг үед тохиолддог)")
else:
    print(three_items)

---
## 📏 АЛХАМ 11: Холбоо хамаарлын дүрмүүдийг гаргах (Association Rules)

**Тайлбар:** Одоо хамгийн чухал хэсэг — **Association Rules** буюу **Холбоо Хамаарлын Дүрмүүд** гаргах.

### Гол хэмжүүрүүд:

| Хэмжүүр | Тайлбар | Формул |
|---------|---------|--------|
| **Support** | Хослол нийт гүйлгээний хэдэн хувьд байна | P(A ∩ B) |
| **Confidence** | A авсан хүн B-г авах магадлал | P(B\|A) = P(A ∩ B) / P(A) |
| **Lift** | Хослол санамсаргүй биш, жинхэнэ хамааралтай эсэх | P(A ∩ B) / (P(A) × P(B)) |

**Lift > 1** бол хоёр бараа хоорондоо жинхэнэ хамааралтай гэсэн үг!

- Lift = 1 → хамааралгүй  
- Lift > 1 → хамт авах магадлал өндөр  
- Lift < 1 → хамт авах магадлал бага

In [ ]:
# Association Rules гаргах
# metric="lift" → lift хэмжүүрийг ашиглах
# min_threshold=1.0 → lift нь 1-ээс их байх ёстой

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# Antecedents ба Consequents-ийн уртыг нэмэх
rules["antecedents_length"] = rules["antecedents"].apply(lambda x: len(x))
rules["consequents_length"] = rules["consequents"].apply(lambda x: len(x))

print(f"✅ Нийт {rules.shape[0]} холбоо хамаарлын дүрэм гарлаа.")
print()
print("Баганын тайлбар:")
print("  antecedents = Шалтгаан бараа (X → хэрэв X авбал)")
print("  consequents = Үр дагавар бараа (→ Y авах магадлал)")
print("  support     = Хослолын нийт тохиолдлын хувь")
print("  confidence  = X авсан хүн Y авах магадлал")
print("  lift        = Хослолын жинхэнэ хамаарлын хэмжүүр")
print()
rules

---
## 📈 АЛХАМ 12: Lift-ээр эрэмбэлэх

**Тайлбар:** Хамгийн хүчтэй холбоо хамаарлыг (Lift хамгийн их) эхлээд харуулна. Lift өндөр байх тусам тухайн хосолсон бараануудын хооронд хүчтэй хамаарал байна.

In [ ]:
# Lift-ээр буурахаар эрэмбэлэх
print("🏆 Хамгийн хүчтэй холбоо хамаарлын дүрмүүд (Lift-ээр эрэмбэлсэн):")
rules.sort_values("lift", ascending=False).head(10)

---
## 📊 АЛХАМ 13: Confidence-ээр эрэмбэлэх

**Тайлбар:** **Confidence** нь "Хэрэглэгч А барааг авбал В барааг авах магадлал хичнээн байна" гэсэн утгатай. Confidence өндөр байх тусам тухайн дүрэм найдвартай гэж үзнэ.

In [ ]:
# Confidence-ээр буурахаар эрэмбэлэх
print("💯 Хамгийн өндөр Confidence-тэй дүрмүүд:")
rules.sort_values("confidence", ascending=False).head(10)

---
## 🔎 АЛХАМ 14: Тодорхой бараануудтай холбоотой дүрмүүдийг шүүх

**Тайлбар:** Зөвхөн `BREAD` (Талх)-тай холбоотой дүрмүүдийг харна. Дэлгүүрийн менежер `"ТАЛХ авсан хүн өөр юу авах вэ?"` гэж асуух үед ашиглана.

In [ ]:
# BREAD-тэй холбоотой дүрмүүд — BREAD шалтгаан болж байгаа
bread_rules = rules[
    rules["antecedents"].apply(lambda x: "BREAD" in x) &
    (rules["antecedents_length"] == 1)
].sort_values("confidence", ascending=False)

print("🍞 'BREAD' авсан хэрэглэгч өөр ямар барааг авах магадлал өндөр вэ?")
print()
if bread_rules.empty:
    print("⚠️ BREAD-тэй холбоотой хангалттай дүрэм олдсонгүй.")
else:
    print(bread_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

In [ ]:
# TEA-тэй холбоотой дүрмүүд
tea_rules = rules[
    rules["antecedents"].apply(lambda x: "TEA" in x) &
    (rules["antecedents_length"] == 1)
].sort_values("confidence", ascending=False)

print("🍵 'TEA' авсан хэрэглэгч өөр ямар барааг авах магадлал өндөр вэ?")
print()
if tea_rules.empty:
    print("⚠️ TEA-тэй холбоотой хангалттай дүрэм олдсонгүй.")
else:
    print(tea_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

In [ ]:
# COFFEE-тэй холбоотой дүрмүүд
coffee_rules = rules[
    rules["antecedents"].apply(lambda x: "COFFEE" in x) &
    (rules["antecedents_length"] == 1)
].sort_values("confidence", ascending=False)

print("☕ 'COFFEE' авсан хэрэглэгч өөр ямар барааг авах магадлал өндөр вэ?")
print()
if coffee_rules.empty:
    print("⚠️ COFFEE-тэй холбоотой хангалттай дүрэм олдсонгүй.")
else:
    print(coffee_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

---
## 📋 АЛХАМ 15: Нийт дүн шинжилгээний хураангуй

**Тайлбар:** Бүх үр дүнгийн хураангуйг нэг газар харна.

In [ ]:
print("=" * 60)
print("📊 НИЙТ ДҮН ШИНЖИЛГЭЭНИЙ ХУРААНГУЙ")
print("=" * 60)
print(f"  📦 Нийт гүйлгээ:              {len(transactions)}")
print(f"  🏷️  Нийт өвөрмөц бараа:        {dataset.shape[1]}")
print(f"  📈 Давтамжтай зүйлсийн багц:  {frequent_itemsets.shape[0]}")
print(f"     - Нэг барааны хослол:      {frequent_itemsets[frequent_itemsets['length']==1].shape[0]}")
print(f"     - Хоёр барааны хослол:     {frequent_itemsets[frequent_itemsets['length']==2].shape[0]}")
print(f"     - Гурван барааны хослол:   {frequent_itemsets[frequent_itemsets['length']==3].shape[0]}")
print(f"  📏 Нийт холбоо хамаарлын дүрэм: {rules.shape[0]}")
print()
print("🏆 Хамгийн өндөр Lift-тэй дүрэм:")
top_rule = rules.sort_values('lift', ascending=False).iloc[0]
print(f"   {set(top_rule['antecedents'])} → {set(top_rule['consequents'])}")
print(f"   Confidence: {top_rule['confidence']:.2%}, Lift: {top_rule['lift']:.3f}")
print()
print("💡 Санал:")
print("   Дэлгүүрийн менежер дээрх бараануудыг ойролцоо байрлуулбал")
print("   хэрэглэгчдийн худалдан авалт нэмэгдэх боломжтой!")
print("=" * 60)

---
## 🎓 ДҮГНЭЛТ

### Энэ дасгалаар юу сурсан бэ?

1. **Apriori алгоритм** нь дэлгүүрийн гүйлгээний өгөгдлөөс хамт зардаг бараануудын хослолыг автоматаар олдог.

2. **Гурван гол хэмжүүр:**
   - **Support** → Хэр их тохиолдож байна?
   - **Confidence** → Нэг барааг авбал нөгөөг авах магадлал хэд?
   - **Lift** → Тэдний хамаарал санамсаргүй биш мөн эсэх?

3. **Практик ашиглалт:**
   - Дэлгүүрийн бараа байршуулалт оновчлох
   - "Та мөн... сонирхож болох" санал болгох систем (Netflix, Amazon)
   - Хямдралын урамшуулал тохируулах

### dataset_2.csv-ийн гол олдвор:
- **BREAD** (Талх) хамгийн их гүйлгээнд тохиолддог бараа
- **BREAD + MILK** хосолсон нь хамгийн давтамжтай хоёр барааны хослол
- **Lift > 1** бүхий бараанууд хоорондоо жинхэнэ хамааралтай байна